In [84]:
pip install requests pandas

In [85]:
!pip -q install pandas requests tqdm

import time
from pathlib import Path

import pandas as pd
import requests
from tqdm.auto import tqdm

print("Entorno preparado correctamente.")

Entorno preparado correctamente.


In [87]:
# ======================================================================
# OBIS -> GOOGLE DRIVE -> JSON
#
# El JSON contiene exactamente:
# scientificName, latitude, longitude, eventDate, year, month, day
#
# CARACTERISTICAS:
# - El nombre del archivo JSON se genera desde ESPECIE.
# - Conserva todos los registros duplicados.
# - No utiliza drop_duplicates().
# - Extrae year, month y day desde eventDate.
# - Utiliza los campos temporales originales de OBIS como respaldo.
# - No inventa meses ni dias que no esten disponibles.
# - Guarda el JSON directamente en Google Drive.
# ======================================================================

import json
import re
import time
import unicodedata
from pathlib import Path

import pandas as pd
import requests
from google.colab import drive


# ======================================================================
# 1. CONFIGURACION
# ======================================================================

# Cambiad solamente esta linea para consultar otra especie.
ESPECIE = "Posidonia oceanica"

URL_OBIS = "https://api.obis.org/v3/occurrence"

# Carpeta de Google Drive donde se guardara el archivo.
NOMBRE_CARPETA_DRIVE = "Space_Apps_Challenge"

# Numero maximo de registros solicitados en cada peticion.
TAMANO_LOTE = 2000

# Numero maximo de registros que se descargaran.
#
# Ejemplos:
#   MAX_REGISTROS = 100       -> prueba con 100 registros
#   MAX_REGISTROS = 10_000    -> descarga hasta 10.000
#   MAX_REGISTROS = None      -> descarga todos los registros
MAX_REGISTROS = 10_000


# ======================================================================
# 2. GENERAR EL NOMBRE DEL ARCHIVO DESDE LA ESPECIE
# ======================================================================

def generar_nombre_archivo(especie):
    """
    Genera un nombre de archivo seguro a partir de la especie.

    Ejemplos:
        Posidonia oceanica -> Posidonia_oceanica.json
        Caretta caretta    -> Caretta_caretta.json
        Thunnus thynnus    -> Thunnus_thynnus.json
    """

    nombre = str(especie).strip()

    if not nombre:
        raise ValueError(
            "ESPECIE no puede estar vacio."
        )

    # Convertir caracteres acentuados a su equivalente basico.
    nombre = unicodedata.normalize(
        "NFKD",
        nombre,
    ).encode(
        "ascii",
        "ignore",
    ).decode("ascii")

    # Sustituir espacios por guiones bajos.
    nombre = re.sub(
        r"\s+",
        "_",
        nombre,
    )

    # Eliminar caracteres no seguros para un nombre de archivo.
    nombre = re.sub(
        r"[^A-Za-z0-9_-]",
        "",
        nombre,
    )

    # Evitar multiples guiones bajos seguidos.
    nombre = re.sub(
        r"_+",
        "_",
        nombre,
    )

    nombre = nombre.strip("_-")

    if not nombre:
        raise ValueError(
            "No se pudo generar un nombre de archivo desde ESPECIE."
        )

    return f"{nombre}.json"


NOMBRE_ARCHIVO = generar_nombre_archivo(
    ESPECIE
)

print(f"Especie seleccionada: {ESPECIE}")
print(f"Nombre del archivo: {NOMBRE_ARCHIVO}")


# ======================================================================
# 3. CONECTAR GOOGLE DRIVE
# ======================================================================

RUTA_DRIVE = Path("/content/drive")
RUTA_MI_UNIDAD = RUTA_DRIVE / "MyDrive"

if RUTA_MI_UNIDAD.exists():
    print("Google Drive ya esta conectado.")
else:
    print("Conectando Google Drive...")
    drive.mount("/content/drive")

if not RUTA_MI_UNIDAD.exists():
    raise RuntimeError(
        "No se ha podido acceder a Google Drive. "
        "Reiniciad el entorno de ejecucion de Colab y volved "
        "a ejecutar esta celda."
    )


# ======================================================================
# 4. CREAR LA CARPETA Y LA RUTA DEL JSON
# ======================================================================

CARPETA_FINAL = (
    RUTA_MI_UNIDAD
    / NOMBRE_CARPETA_DRIVE
)

CARPETA_FINAL.mkdir(
    parents=True,
    exist_ok=True,
)

RUTA_JSON = (
    CARPETA_FINAL
    / NOMBRE_ARCHIVO
)

print(
    f"\nEl archivo JSON se guardara en:\n"
    f"{RUTA_JSON}\n"
)


# ======================================================================
# 5. CAMPOS SOLICITADOS A OBIS
# ======================================================================

# id se solicita para la paginacion, pero no se guarda
# en el JSON final.
CAMPOS_OBIS = [
    "id",
    "scientificName",
    "decimalLatitude",
    "decimalLongitude",
    "eventDate",
    "year",
    "month",
    "day",
]


# ======================================================================
# 6. CONSULTAR LA API DE OBIS
# ======================================================================

def consultar_obis(parametros, max_intentos=5):
    """
    Consulta OBIS con reintentos automaticos.
    """

    cabeceras = {
        "Accept": "application/json",
        "User-Agent": (
            "NASA-Space-Apps-Earth-Trend-Detective/1.0 "
            "(educational hackathon project)"
        ),
    }

    ultimo_error = None

    for intento in range(1, max_intentos + 1):
        try:
            respuesta = requests.get(
                URL_OBIS,
                params=parametros,
                headers=cabeceras,
                timeout=120,
            )

            # Estos errores suelen ser temporales.
            if (
                respuesta.status_code == 429
                or respuesta.status_code >= 500
            ):
                espera = min(
                    60,
                    2 ** intento,
                )

                print(
                    f"Error temporal {respuesta.status_code}. "
                    f"Reintentando en {espera} segundos..."
                )

                time.sleep(espera)
                continue

            respuesta.raise_for_status()

            return respuesta.json()

        except requests.RequestException as error:
            ultimo_error = error

            if intento < max_intentos:
                espera = min(
                    60,
                    2 ** intento,
                )

                print(
                    f"Problema de conexion: {error}\n"
                    f"Reintentando en {espera} segundos..."
                )

                time.sleep(espera)

    raise RuntimeError(
        "No se pudo obtener una respuesta valida de OBIS."
    ) from ultimo_error


# ======================================================================
# 7. EXTRAER LOS REGISTROS DE LA RESPUESTA
# ======================================================================

def extraer_registros(respuesta_json):
    """
    Extrae la lista de observaciones de la respuesta de OBIS.
    """

    if isinstance(respuesta_json, list):
        return respuesta_json

    if not isinstance(respuesta_json, dict):
        raise ValueError(
            "OBIS ha devuelto un formato inesperado."
        )

    if isinstance(
        respuesta_json.get("results"),
        list,
    ):
        return respuesta_json["results"]

    if isinstance(
        respuesta_json.get("data"),
        list,
    ):
        return respuesta_json["data"]

    raise ValueError(
        "No se encontraron observaciones en la respuesta de OBIS. "
        f"Claves recibidas: {list(respuesta_json.keys())}"
    )


# ======================================================================
# 8. DESCARGAR LAS OBSERVACIONES
# ======================================================================

def descargar_observaciones(
    especie,
    tamano_lote=2000,
    max_registros=None,
):
    """
    Descarga observaciones desde OBIS utilizando paginacion
    mediante el cursor 'after'.

    Todos los registros recibidos se conservan, incluidos
    los registros duplicados.
    """

    registros_totales = []
    cursor_after = None
    numero_pagina = 1

    while True:
        tamano_peticion = tamano_lote

        if max_registros is not None:
            registros_restantes = (
                max_registros
                - len(registros_totales)
            )

            if registros_restantes <= 0:
                break

            tamano_peticion = min(
                tamano_lote,
                registros_restantes,
            )

        parametros = {
            "scientificname": especie,
            "size": tamano_peticion,
            "fields": ",".join(CAMPOS_OBIS),
        }

        if cursor_after is not None:
            parametros["after"] = cursor_after

        respuesta_json = consultar_obis(
            parametros
        )

        registros_pagina = extraer_registros(
            respuesta_json
        )

        if not registros_pagina:
            break

        # Se conservan todos los registros.
        registros_totales.extend(
            registros_pagina
        )

        print(
            f"Pagina {numero_pagina}: "
            f"{len(registros_pagina):,} registros | "
            f"Total acumulado: {len(registros_totales):,}"
        )

        if max_registros is not None:
            if len(registros_totales) >= max_registros:
                registros_totales = registros_totales[
                    :max_registros
                ]
                break

        # Si llegan menos registros de los solicitados,
        # hemos alcanzado la ultima pagina.
        if len(registros_pagina) < tamano_peticion:
            break

        ultimo_id = registros_pagina[-1].get(
            "id"
        )

        if ultimo_id is None:
            raise RuntimeError(
                "El ultimo registro no contiene 'id'. "
                "No se puede continuar la paginacion."
            )

        if ultimo_id == cursor_after:
            raise RuntimeError(
                "OBIS ha repetido el cursor. "
                "La descarga se detiene para evitar un bucle."
            )

        cursor_after = ultimo_id
        numero_pagina += 1

        # Pausa para no saturar la API.
        time.sleep(0.3)

    return pd.DataFrame(
        registros_totales
    )


# ======================================================================
# 9. INTERPRETAR eventDate
# ======================================================================

def resultado_fecha_vacia():
    """
    Devuelve un resultado vacio para una fecha no interpretable.
    """

    return pd.Series(
        [pd.NA, pd.NA, pd.NA],
        index=[
            "year_event",
            "month_event",
            "day_event",
        ],
    )


def fecha_valida(year, month, day):
    """
    Comprueba si una combinacion de año, mes y dia es valida.
    """

    try:
        pd.Timestamp(
            year=int(year),
            month=int(month),
            day=int(day),
        )

        return True

    except (TypeError, ValueError):
        return False


def extraer_fecha_event_date(valor):
    """
    Extrae year, month y day desde eventDate.

    Formatos contemplados:
        2020-07-15
        2020-07-15T10:30:00Z
        2020-07-15T10:30:00+02:00
        2020-07-15/2020-07-20
        2020-07
        2020
        2020/07/15
        15/07/2020
        15-07-2020

    En un intervalo se utiliza la fecha inicial.

    Si solo existe el año:
        year = año
        month = null
        day = null

    Si solo existen año y mes:
        year = año
        month = mes
        day = null
    """

    if valor is None:
        return resultado_fecha_vacia()

    try:
        if pd.isna(valor):
            return resultado_fecha_vacia()
    except (TypeError, ValueError):
        pass

    texto = str(valor).strip()

    if not texto:
        return resultado_fecha_vacia()

    columnas = [
        "year_event",
        "month_event",
        "day_event",
    ]

    # ----------------------------------------------------------
    # YYYY-MM-DD o YYYY/MM/DD
    # ----------------------------------------------------------

    coincidencia = re.search(
        r"(?<!\d)"
        r"(\d{4})"
        r"[-/]"
        r"(\d{1,2})"
        r"[-/]"
        r"(\d{1,2})"
        r"(?!\d)",
        texto,
    )

    if coincidencia:
        year = int(coincidencia.group(1))
        month = int(coincidencia.group(2))
        day = int(coincidencia.group(3))

        if fecha_valida(
            year,
            month,
            day,
        ):
            return pd.Series(
                [year, month, day],
                index=columnas,
            )

    # ----------------------------------------------------------
    # DD/MM/YYYY o DD-MM-YYYY
    # ----------------------------------------------------------

    coincidencia = re.search(
        r"(?<!\d)"
        r"(\d{1,2})"
        r"[-/]"
        r"(\d{1,2})"
        r"[-/]"
        r"(\d{4})"
        r"(?!\d)",
        texto,
    )

    if coincidencia:
        day = int(coincidencia.group(1))
        month = int(coincidencia.group(2))
        year = int(coincidencia.group(3))

        if fecha_valida(
            year,
            month,
            day,
        ):
            return pd.Series(
                [year, month, day],
                index=columnas,
            )

    # ----------------------------------------------------------
    # YYYY-MM o YYYY/MM
    # ----------------------------------------------------------

    coincidencia = re.search(
        r"(?<!\d)"
        r"(\d{4})"
        r"[-/]"
        r"(\d{1,2})"
        r"(?![-/\d])",
        texto,
    )

    if coincidencia:
        year = int(coincidencia.group(1))
        month = int(coincidencia.group(2))

        if (
            1800 <= year <= 2100
            and 1 <= month <= 12
        ):
            return pd.Series(
                [year, month, pd.NA],
                index=columnas,
            )

    # ----------------------------------------------------------
    # Solamente YYYY
    # ----------------------------------------------------------

    coincidencia = re.search(
        r"(?<!\d)"
        r"(18\d{2}|19\d{2}|20\d{2}|2100)"
        r"(?!\d)",
        texto,
    )

    if coincidencia:
        year = int(
            coincidencia.group(1)
        )

        return pd.Series(
            [year, pd.NA, pd.NA],
            index=columnas,
        )

    # ----------------------------------------------------------
    # Intento adicional con pandas
    # ----------------------------------------------------------

    try:
        texto_parseable = texto

        # Para un intervalo ISO, utilizar la fecha inicial.
        intervalo_iso = re.match(
            r"^\s*(\d{4}-\d{1,2}-\d{1,2})/",
            texto,
        )

        if intervalo_iso:
            texto_parseable = intervalo_iso.group(1)

        fecha = pd.to_datetime(
            texto_parseable,
            errors="coerce",
            utc=True,
        )

        if pd.notna(fecha):
            return pd.Series(
                [
                    int(fecha.year),
                    int(fecha.month),
                    int(fecha.day),
                ],
                index=columnas,
            )

    except Exception:
        pass

    return resultado_fecha_vacia()


# ======================================================================
# 10. DESCARGAR LOS DATOS DESDE OBIS
# ======================================================================

print(
    f"\nDescargando observaciones de {ESPECIE}...\n"
)

df_obis = descargar_observaciones(
    especie=ESPECIE,
    tamano_lote=TAMANO_LOTE,
    max_registros=MAX_REGISTROS,
)

if df_obis.empty:
    raise RuntimeError(
        f"OBIS no ha devuelto observaciones para {ESPECIE}."
    )

print(
    f"\nRegistros descargados desde OBIS: "
    f"{len(df_obis):,}"
)


# ======================================================================
# 11. ASEGURAR QUE EXISTAN LAS COLUMNAS
# ======================================================================

columnas_origen = [
    "scientificName",
    "decimalLatitude",
    "decimalLongitude",
    "eventDate",
    "year",
    "month",
    "day",
]

for columna in columnas_origen:
    if columna not in df_obis.columns:
        df_obis[columna] = pd.NA


# ======================================================================
# 12. EXTRAER LA FECHA DESDE eventDate
# ======================================================================

print(
    "\nInterpretando los formatos de eventDate..."
)

fechas_extraidas = (
    df_obis["eventDate"]
    .apply(extraer_fecha_event_date)
)

df_trabajo = pd.concat(
    [
        df_obis.reset_index(drop=True),
        fechas_extraidas.reset_index(drop=True),
    ],
    axis=1,
)


# ======================================================================
# 13. COMBINAR eventDate CON YEAR, MONTH Y DAY ORIGINALES
# ======================================================================

# eventDate tiene prioridad.
# Los campos originales de OBIS se utilizan como respaldo.

year_original = pd.to_numeric(
    df_trabajo["year"],
    errors="coerce",
)

month_original = pd.to_numeric(
    df_trabajo["month"],
    errors="coerce",
)

day_original = pd.to_numeric(
    df_trabajo["day"],
    errors="coerce",
)

year_event = pd.to_numeric(
    df_trabajo["year_event"],
    errors="coerce",
)

month_event = pd.to_numeric(
    df_trabajo["month_event"],
    errors="coerce",
)

day_event = pd.to_numeric(
    df_trabajo["day_event"],
    errors="coerce",
)

df_trabajo["year_final"] = (
    year_event.fillna(year_original)
)

df_trabajo["month_final"] = (
    month_event.fillna(month_original)
)

df_trabajo["day_final"] = (
    day_event.fillna(day_original)
)


# ======================================================================
# 14. CREAR EL DATAFRAME FINAL
# ======================================================================

COLUMNAS_FINALES = [
    "scientificName",
    "latitude",
    "longitude",
    "eventDate",
    "year",
    "month",
    "day",
]

df_final = pd.DataFrame(
    {
        "scientificName": (
            df_trabajo["scientificName"]
        ),
        "latitude": (
            df_trabajo["decimalLatitude"]
        ),
        "longitude": (
            df_trabajo["decimalLongitude"]
        ),
        "eventDate": (
            df_trabajo["eventDate"]
        ),
        "year": (
            df_trabajo["year_final"]
        ),
        "month": (
            df_trabajo["month_final"]
        ),
        "day": (
            df_trabajo["day_final"]
        ),
    }
)

# Garantizar que solo se incluyan estas columnas.
df_final = df_final[
    COLUMNAS_FINALES
].copy()


# ======================================================================
# 15. CONVERTIR LOS TIPOS
# ======================================================================

for columna in [
    "latitude",
    "longitude",
    "year",
    "month",
    "day",
]:
    df_final[columna] = pd.to_numeric(
        df_final[columna],
        errors="coerce",
    )


# ======================================================================
# 16. VALIDAR LAS COORDENADAS
# ======================================================================

# Se eliminan solamente observaciones sin especie o coordenadas.
# Los duplicados se conservan.

df_final = df_final.dropna(
    subset=[
        "scientificName",
        "latitude",
        "longitude",
    ]
)

df_final = df_final[
    df_final["latitude"].between(-90, 90)
    & df_final["longitude"].between(-180, 180)
].copy()


# ======================================================================
# 17. VALIDAR YEAR, MONTH Y DAY
# ======================================================================

df_final.loc[
    df_final["year"].notna()
    & ~df_final["year"].between(1800, 2100),
    "year",
] = pd.NA

df_final.loc[
    df_final["month"].notna()
    & ~df_final["month"].between(1, 12),
    "month",
] = pd.NA

df_final.loc[
    df_final["day"].notna()
    & ~df_final["day"].between(1, 31),
    "day",
] = pd.NA


# ======================================================================
# 18. VALIDAR LAS FECHAS COMPLETAS
# ======================================================================

filas_con_fecha_completa = (
    df_final["year"].notna()
    & df_final["month"].notna()
    & df_final["day"].notna()
)

indices_fecha_completa = df_final.index[
    filas_con_fecha_completa
]

for indice in indices_fecha_completa:
    try:
        pd.Timestamp(
            year=int(
                df_final.at[indice, "year"]
            ),
            month=int(
                df_final.at[indice, "month"]
            ),
            day=int(
                df_final.at[indice, "day"]
            ),
        )

    except ValueError:
        # Si el dia es imposible, se conserva el año y el mes.
        df_final.at[
            indice,
            "day",
        ] = pd.NA


# ======================================================================
# 19. PREPARAR LOS TIPOS FINALES
# ======================================================================

for columna in [
    "year",
    "month",
    "day",
]:
    df_final[columna] = (
        df_final[columna]
        .astype("Int64")
    )

# No se eliminan duplicados.
df_final = df_final.reset_index(
    drop=True
)

# Proteccion final contra columnas adicionales.
df_final = df_final[
    COLUMNAS_FINALES
]


# ======================================================================
# 20. CONVERTIR LOS DATOS A REGISTROS JSON
# ======================================================================

registros_json = []

for registro in df_final.to_dict(
    orient="records"
):
    registro_limpio = {}

    for clave, valor in registro.items():
        if valor is None:
            registro_limpio[clave] = None

        else:
            try:
                valor_vacio = pd.isna(valor)
            except (TypeError, ValueError):
                valor_vacio = False

            if valor_vacio:
                registro_limpio[clave] = None

            elif clave in [
                "year",
                "month",
                "day",
            ]:
                registro_limpio[clave] = int(
                    valor
                )

            elif clave in [
                "latitude",
                "longitude",
            ]:
                registro_limpio[clave] = float(
                    valor
                )

            else:
                registro_limpio[clave] = str(
                    valor
                )

    registros_json.append(
        registro_limpio
    )


# ======================================================================
# 21. GUARDAR EL JSON EN GOOGLE DRIVE
# ======================================================================

# Guardar primero en un archivo temporal evita dejar un JSON
# incompleto si la escritura se interrumpe.
RUTA_TEMPORAL = Path(
    str(RUTA_JSON) + ".tmp"
)

with open(
    RUTA_TEMPORAL,
    mode="w",
    encoding="utf-8",
) as archivo_json:
    json.dump(
        registros_json,
        archivo_json,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    )

# Reemplazar el archivo anterior, si existe.
RUTA_TEMPORAL.replace(
    RUTA_JSON
)


# ======================================================================
# 22. COMPROBAR EL JSON GUARDADO
# ======================================================================

if not RUTA_JSON.exists():
    raise FileNotFoundError(
        f"No se ha creado el archivo:\n{RUTA_JSON}"
    )

with open(
    RUTA_JSON,
    mode="r",
    encoding="utf-8",
) as archivo_json:
    datos_comprobacion = json.load(
        archivo_json
    )

if not isinstance(
    datos_comprobacion,
    list,
):
    raise RuntimeError(
        "El archivo creado no contiene una lista JSON."
    )

if len(datos_comprobacion) != len(df_final):
    raise RuntimeError(
        "El numero de registros del JSON no coincide "
        "con el numero de observaciones procesadas."
    )

if datos_comprobacion:
    campos_guardados = list(
        datos_comprobacion[0].keys()
    )

    if campos_guardados != COLUMNAS_FINALES:
        raise RuntimeError(
            "El JSON no contiene exactamente los campos esperados.\n"
            f"Campos encontrados: {campos_guardados}"
        )


# ======================================================================
# 23. CALCULAR EL RESUMEN
# ======================================================================

total_descargados = len(
    df_obis
)

total_guardados = len(
    datos_comprobacion
)

con_event_date = (
    df_final["eventDate"]
    .notna()
    .sum()
)

con_year = (
    df_final["year"]
    .notna()
    .sum()
)

con_month = (
    df_final["month"]
    .notna()
    .sum()
)

con_day = (
    df_final["day"]
    .notna()
    .sum()
)

con_fecha_completa = (
    df_final[
        ["year", "month", "day"]
    ]
    .notna()
    .all(axis=1)
    .sum()
)

sin_fecha_utilizable = (
    df_final[
        ["year", "month", "day"]
    ]
    .isna()
    .all(axis=1)
    .sum()
)

tamano_mb = (
    RUTA_JSON.stat().st_size
    / 1_000_000
)


# ======================================================================
# 24. MOSTRAR EL RESULTADO
# ======================================================================

print("\n" + "=" * 72)
print("JSON CREADO CORRECTAMENTE EN GOOGLE DRIVE")
print("=" * 72)

print(
    f"\nEspecie: {ESPECIE}"
)

print(
    f"Nombre del archivo: {NOMBRE_ARCHIVO}"
)

print(
    f"\nRuta:\n{RUTA_JSON}"
)

print(
    f"\nRegistros descargados desde OBIS: "
    f"{total_descargados:,}"
)

print(
    f"Registros guardados en el JSON: "
    f"{total_guardados:,}"
)

print(
    f"Tamaño del archivo: "
    f"{tamano_mb:.2f} MB"
)

print(
    "\nTodos los registros duplicados se han conservado."
)

print(
    f"\nObservaciones con eventDate: "
    f"{con_event_date:,}"
)

print(
    f"Observaciones con año: "
    f"{con_year:,}"
)

print(
    f"Observaciones con mes: "
    f"{con_month:,}"
)

print(
    f"Observaciones con dia: "
    f"{con_day:,}"
)

print(
    f"Observaciones con fecha completa: "
    f"{con_fecha_completa:,}"
)

print(
    f"Observaciones sin fecha utilizable: "
    f"{sin_fecha_utilizable:,}"
)

print("\nCampos guardados:")
print(COLUMNAS_FINALES)

print("\nPrimer registro del JSON:")

if datos_comprobacion:
    print(
        json.dumps(
            datos_comprobacion[0],
            ensure_ascii=False,
            indent=2,
        )
    )
else:
    print(
        "El JSON no contiene registros."
    )

Especie seleccionada: Posidonia oceanica
Nombre del archivo: Posidonia_oceanica.json
Google Drive ya esta conectado.

El archivo JSON se guardara en:
/content/drive/MyDrive/Space_Apps_Challenge/Posidonia_oceanica.json


Descargando observaciones de Posidonia oceanica...

Pagina 1: 2,000 registros | Total acumulado: 2,000
Pagina 2: 2,000 registros | Total acumulado: 4,000
Pagina 3: 2,000 registros | Total acumulado: 6,000
Pagina 4: 2,000 registros | Total acumulado: 8,000
Pagina 5: 2,000 registros | Total acumulado: 10,000

Registros descargados desde OBIS: 10,000

Interpretando los formatos de eventDate...

JSON CREADO CORRECTAMENTE EN GOOGLE DRIVE

Especie: Posidonia oceanica
Nombre del archivo: Posidonia_oceanica.json

Ruta:
/content/drive/MyDrive/Space_Apps_Challenge/Posidonia_oceanica.json

Registros descargados desde OBIS: 10,000
Registros guardados en el JSON: 10,000
Tamaño del archivo: 2.06 MB

Todos los registros duplicados se han conservado.

Observaciones con eventDate: 9,988